# War Room Catalyst Review
**Charlie OS 5.0 | War Room v10.0**

Run all cells top to bottom to refresh. Each section is independent — jump to any one you need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

DATA_DIR = Path("..") / "My Data"
CATALYST_CSV = DATA_DIR / "catalyst_log.csv"
OUTCOMES_CSV = DATA_DIR / "outcomes.csv"

def load_data():
    cat = pd.read_csv(CATALYST_CSV)
    out = pd.read_csv(OUTCOMES_CSV) if OUTCOMES_CSV.exists() else pd.DataFrame()
    if not out.empty and "catalyst_id" in out.columns:
        merged = cat.merge(out, left_on="id", right_on="catalyst_id", how="left", suffixes=("", "_out"))
    else:
        merged = cat.copy()
        for col in ["return_1d","return_3d","return_5d","return_10d","return_20d",
                     "price_at_analysis","catalyst_resolved","resolution_notes"]:
            if col not in merged.columns:
                merged[col] = np.nan
    return merged

# Color map for conviction tags
CONVICTION_COLORS = {"HIGH": "#2ecc71", "DEVELOPING": "#f39c12", "WEAK": "#e74c3c"}
ENGINE_COLORS = {"F": "#3498db", "M": "#9b59b6", "N": "#e67e22",
                 "F+M": "#1abc9c", "F+N": "#2980b9", "ALL THREE": "#e74c3c"}

def color_score(val):
    if pd.isna(val): return ''
    if val >= 8: return 'background-color: #1a4a2e; color: #2ecc71'
    if val >= 5: return 'background-color: #4a3a1a; color: #f39c12'
    return 'background-color: #4a1a1a; color: #e74c3c'

def color_return(val):
    if pd.isna(val): return ''
    return 'color: #2ecc71' if val > 0 else 'color: #e74c3c'

plt.style.use('dark_background')
print(f"CSV: {CATALYST_CSV.resolve()}")
print(f"Exists: {CATALYST_CSV.exists()}")

---
## 1. Dashboard Summary

In [ ]:
df = load_data()

total = len(df)
high = (df["conviction_tag"] == "HIGH").sum()
developing = (df["conviction_tag"] == "DEVELOPING").sum()
weak = (df["conviction_tag"] == "WEAK").sum()
earnings_catalysts = df["is_earnings"].sum() if "is_earnings" in df.columns else 0
seasons = sorted(df[df["earnings_season"].notna()]["earnings_season"].unique()) if "earnings_season" in df.columns else []
with_outcomes = df["return_5d"].notna().sum()
avg_score = df["score_total"].mean()

print("=" * 50)
print("  WAR ROOM CATALYST TRACKER -- SUMMARY")
print("=" * 50)
print(f"  Total logged          : {total}")
print(f"  With 5-day outcomes   : {int(with_outcomes)}")
print(f"  Avg score             : {avg_score:.1f}/10" if pd.notna(avg_score) else "  Avg score             : n/a")
print(f"  HIGH conviction       : {high}")
print(f"  DEVELOPING            : {developing}")
print(f"  WEAK                  : {weak}")
print(f"  Earnings catalysts    : {int(earnings_catalysts)}")
print(f"  Seasons tracked       : {', '.join(seasons) if seasons else 'none yet'}")

---
## 2. Full Catalyst Log

In [ ]:
df = load_data()

display_df = df[['id','analysis_date','ticker','score_total','conviction_tag',
                 'catalyst_type','catalyst_engine','sector','earnings_season',
                 'macro_context','catalyst_direction','the_one_variable',
                 'return_1d','return_3d','return_5d','return_10d','return_20d',
                 'catalyst_resolved']].copy()
display_df.columns = ['id','date','ticker','score','conviction','type','engine','sector',
                       'season','macro','direction','one_variable',
                       'r1d','r3d','r5d','r10d','r20d','resolved']
display_df = display_df.sort_values(['date','id'], ascending=[False, False])

styled = (
    display_df.style
    .map(color_score, subset=['score'])
    .map(color_return, subset=['r1d','r3d','r5d','r10d','r20d'])
    .format({
        'score': '{:.1f}',
        'r1d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r3d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r5d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r10d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        'r20d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
    })
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#1a1a2e'), ('color', '#aaa')]}])
)

display(styled)

---
## 3. Earnings Season Scoreboard

In [ ]:
# Change SEASON to focus on a specific one, or leave None for latest
SEASON = None  # e.g. "Q1-2026"

df = load_data()
earnings = df[df["is_earnings"] == 1]

if earnings.empty:
    print("No earnings catalysts logged yet.")
else:
    if SEASON is None:
        SEASON = earnings.sort_values("analysis_date", ascending=False).iloc[0]["earnings_season"]

    es = earnings[earnings["earnings_season"] == SEASON].sort_values("score_total", ascending=False)
    es_display = es[['ticker','analysis_date','score_total','conviction_tag',
                      'catalyst_type_label','catalyst_engine','macro_context',
                      'return_1d','return_3d','return_5d','return_10d']].copy()
    es_display.columns = ['ticker','date','score','conviction','type','engine','macro',
                           'r1d','r3d','r5d','r10d']

    print(f"Earnings Season: {SEASON}  |  {len(es_display)} catalysts")
    if not es_display.empty:
        r5d_valid = es_display['r5d'].dropna()
        win_rate = (r5d_valid > 0).sum() / len(r5d_valid) * 100 if len(r5d_valid) > 0 else None
        avg_5d = r5d_valid.mean()
        print(f"Avg score: {es_display['score'].mean():.1f} | Avg 5d return: {avg_5d:+.2f}%" if pd.notna(avg_5d) else f"Avg score: {es_display['score'].mean():.1f} | 5d returns: pending")
        if win_rate is not None:
            print(f"5d win rate: {win_rate:.0f}%")

    styled_es = (
        es_display.style
        .map(color_score, subset=['score'])
        .map(color_return, subset=['r1d','r3d','r5d','r10d'])
        .format({
            'score': '{:.1f}',
            'r1d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'r3d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'r5d':  lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
            'r10d': lambda x: f"{x:+.2f}%" if pd.notna(x) else '-',
        })
    )
    display(styled_es)

---
## 4. Performance Charts

In [ ]:
df = load_data()
perf = df[df["return_5d"].notna()].copy()

if perf.empty:
    print("No outcome data yet. Log some outcomes first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor('#0d0d0d')

    # -- Chart 1: Avg return by conviction tier --
    ax = axes[0]
    ax.set_facecolor('#1a1a1a')
    tiers = ['HIGH', 'DEVELOPING', 'WEAK']
    windows = ['return_1d', 'return_3d', 'return_5d', 'return_10d']
    labels = ['1d', '3d', '5d', '10d']
    for tier in tiers:
        subset = perf[perf['conviction_tag'] == tier]
        if not subset.empty:
            avgs = [subset[w].mean() for w in windows]
            ax.plot(labels, avgs, marker='o', label=tier,
                    color=CONVICTION_COLORS.get(tier, 'white'), linewidth=2)
    ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
    ax.set_title('Avg Return by Conviction Tier', color='white')
    ax.set_ylabel('Return %', color='white')
    ax.tick_params(colors='#aaa')
    ax.legend(facecolor='#1a1a1a', labelcolor='white')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:+.1f}%'))

    # -- Chart 2: Avg 5d return by catalyst engine --
    ax2 = axes[1]
    ax2.set_facecolor('#1a1a1a')
    eng_data = perf.groupby('catalyst_engine')['return_5d'].agg(['mean', 'count']).reset_index()
    eng_data = eng_data.sort_values('mean', ascending=False)
    colors = [ENGINE_COLORS.get(e, '#888') for e in eng_data['catalyst_engine']]
    bars = ax2.bar(eng_data['catalyst_engine'], eng_data['mean'], color=colors, alpha=0.85)
    ax2.axhline(0, color='#555', linewidth=0.8, linestyle='--')
    for bar, count in zip(bars, eng_data['count']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'n={count}', ha='center', va='bottom', color='#aaa', fontsize=8)
    ax2.set_title('Avg 5d Return by Engine', color='white')
    ax2.set_ylabel('5d Return %', color='white')
    ax2.tick_params(colors='#aaa')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:+.1f}%'))

    # -- Chart 3: Score distribution histogram --
    ax3 = axes[2]
    ax3.set_facecolor('#1a1a1a')
    scores = df['score_total'].dropna()
    ax3.hist(scores, bins=20, color='#3498db', alpha=0.8, edgecolor='#1a1a1a')
    ax3.axvline(8, color='#2ecc71', linewidth=1.5, linestyle='--', label='HIGH threshold')
    ax3.axvline(5, color='#f39c12', linewidth=1.5, linestyle='--', label='DEVELOPING threshold')
    ax3.set_title('Score Distribution', color='white')
    ax3.set_xlabel('Catalyst Score', color='white')
    ax3.set_ylabel('Count', color='white')
    ax3.tick_params(colors='#aaa')
    ax3.legend(facecolor='#1a1a1a', labelcolor='white')

    plt.tight_layout()
    plt.show()

---
## 5. Performance by Catalyst Type

In [ ]:
df = load_data()
perf = df[df["return_5d"].notna()].copy()

if perf.empty:
    print("No outcome data yet.")
else:
    by_type = perf.groupby('catalyst_type').agg(
        count=('score_total', 'size'),
        avg_score=('score_total', 'mean'),
        avg_1d=('return_1d', 'mean'),
        avg_3d=('return_3d', 'mean'),
        avg_5d=('return_5d', 'mean'),
        avg_10d=('return_10d', 'mean'),
    ).reset_index().sort_values('avg_5d', ascending=False)

    styled = by_type.style.format({
        'avg_score': '{:.1f}',
        'avg_1d': '{:+.2f}%', 'avg_3d': '{:+.2f}%',
        'avg_5d': '{:+.2f}%', 'avg_10d': '{:+.2f}%',
    }).applymap(color_return, subset=['avg_1d','avg_3d','avg_5d','avg_10d']
    ).set_properties(**{
        'background-color': '#1a1a1a', 'color': 'white', 'border': '1px solid #333'
    })
    display(styled)

---
## 6. Ticker Lookup
Change `TICKER` to any symbol you've logged.

In [ ]:
TICKER = "NVDA"  # <-- change this

df = load_data()
tkr = df[df['ticker'] == TICKER].sort_values('analysis_date', ascending=False)

if tkr.empty:
    print(f"No entries for {TICKER}")
else:
    cols = ['analysis_date', 'catalyst_type', 'catalyst_engine', 'score_total',
            'conviction_tag', 'the_one_variable', 'return_1d', 'return_5d', 'return_10d']
    show = tkr[[c for c in cols if c in tkr.columns]]
    styled = show.style.format({
        'score_total': '{:.1f}',
        'return_1d': lambda x: f'{x:+.2f}%' if pd.notna(x) else '—',
        'return_5d': lambda x: f'{x:+.2f}%' if pd.notna(x) else '—',
        'return_10d': lambda x: f'{x:+.2f}%' if pd.notna(x) else '—',
    }).applymap(color_score, subset=['score_total']
    ).set_properties(**{
        'background-color': '#1a1a1a', 'color': 'white', 'border': '1px solid #333'
    })
    display(styled)

---
## 7. Score vs 5-Day Return Scatter

In [ ]:
df = load_data()
scatter = df[df["return_5d"].notna()].copy()

if scatter.empty:
    print("No outcome data yet.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('#0d0d0d')
    ax.set_facecolor('#1a1a1a')

    colors = [CONVICTION_COLORS.get(c, 'white') for c in scatter['conviction_tag']]
    ax.scatter(scatter['score_total'], scatter['return_5d'],
               c=colors, s=60, alpha=0.8, edgecolors='#333')
    ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
    ax.axvline(8, color='#2ecc71', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(5, color='#f39c12', linewidth=0.8, linestyle='--', alpha=0.5)

    ax.set_xlabel('Catalyst Score', color='white')
    ax.set_ylabel('5d Return %', color='white')
    ax.set_title('Score vs 5d Forward Return', color='white')
    ax.tick_params(colors='#aaa')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:+.1f}%'))

    # legend
    for tier, color in CONVICTION_COLORS.items():
        ax.scatter([], [], c=color, label=tier, s=40)
    ax.legend(facecolor='#1a1a1a', labelcolor='white')

    plt.tight_layout()
    plt.show()

---
## 8. Pending Outcomes
Catalysts that have been logged but still need price follow-through.

In [ ]:
df = load_data()
pending = df[df["return_5d"].isna()].copy()

if pending.empty:
    print("✅ All outcomes fetched — nothing pending.")
else:
    cols = ['id', 'analysis_date', 'ticker', 'catalyst_type', 'score_total',
            'conviction_tag', 'the_one_variable']
    show = pending[[c for c in cols if c in pending.columns]].sort_values('analysis_date', ascending=False)
    print(f"⏳ {len(show)} catalysts pending outcome data:\n")
    styled = show.style.format({
        'score_total': '{:.1f}',
    }).applymap(color_score, subset=['score_total']
    ).set_properties(**{
        'background-color': '#1a1a1a', 'color': 'white', 'border': '1px solid #333'
    })
    display(styled)
    print(f"\nRun: python scripts/catalyst_score_log/fetch_returns.py --all")